# TenderWin · Лід і мейл генератор v1.5 — БОЙОВИЙ РЕЖИМ

Робоча тека — **`1 Lead&Mail generator`**. Вісім клітинок, згори вниз;
девʼята — перевірка самого скрипта, за бажанням. Код редагувати не треба.

**Що робить:** шукає в Prozorro відхилення за поточну добу по всіх галузях і
всіх закупівлях, складає перший лист (V10) **на адресу компанії з Prozorro**,
робочу картку закупівлі, а для кожного відхилення — **службову картку і
протокол (рішення) замовника** у `Мій диск / 0 Cases / <ID закупівлі> <ЄДРПОУ>`
(на компʼютері це `G:\Мій диск\0 Cases`).

**Бойовий режим (`M.MODE = "LIVE"`, клітинка 5):**
- клітинка 6 лише складає листи у файли — нічого не надсилає;
- клітинка 7 надсилає **клієнтам** — одразу, без запитань. Тому «Виконати
  всі» розішле листи відразу після складання: щоб спершу переглянути листи,
  виконуйте клітинки по одній (1 → 6), а клітинку 7 — після перегляду;
- усі листи до 08.10.2026 були тестовими — бойового листа вони не займають.
  Одна компанія отримує один бойовий перший лист, назавжди.

Потрібні чотири файли: `tenderwin_lead_mail.py` (1.5.0), `tenderwin_lead_engine.py`,
`lead_machine_v1.py`, `tenderwin_service_cards.py`. Якщо їх ще немає на
Диску — клітинка 2 запропонує завантажити їх із компʼютера.

## КЛІТИНКА 1 · Встановлення бібліотек

In [ ]:
!pip -q install google-api-python-client google-auth google-auth-oauthlib python-docx requests pdfplumber pypdf openpyxl
print("Готово. Далі — клітинка 2.")

## КЛІТИНКА 2 · Google Диск, робоча тека і файли

Монтує Диск і працює з текою **`1 Lead&Mail generator`** (назва — у рядку
`РОБОЧА_ТЕКА`). Там мають лежати чотири файли: `tenderwin_lead_mail.py`,
`tenderwin_lead_engine.py`, `lead_machine_v1.py`, `tenderwin_service_cards.py`.
Якщо якогось бракує — відкриється вікно вибору файлів на компʼютері.

**Перший запуск у новій теці:** база `tenderwin_lead_mail.db` (кому вже
писали), `leads.db` і токен `.gmail_token.json` копіюються сюди з попередньої
теки `TENDERWIN` — **один раз**. Після цього в робочій теці зʼявляється
позначка `.перенесено_з_попередньої_теки.json`, і більше нічого не копіюється.
У `TENDERWIN` нічого не видаляється і не змінюється.

In [ ]:
from google.colab import drive
import json, os, re, shutil, sqlite3, sys, time
from pathlib import Path

НОУТБУК = '1.5.0'
print(f"Ноутбук TenderWin Lead & Mail {НОУТБУК}")
ГОТОВО = False      # стане True лише тоді, коли ця клітинка дійде до кінця
M = E = None        # модулі заново завантажують клітинки 3 і 4 — з робочої теки
НАЛАШТОВАНО = False  # клітинку 5 треба виконати після клітинки 4

drive.mount('/content/drive')

КОРІНЬ = '/content/drive/MyDrive'
#: Робоча тека: скрипти, ноутбук, база, листи. Змінювати назву — лише тут.
РОБОЧА_ТЕКА = '1 Lead&Mail generator'
#: Звідки ОДИН раз перенести базу й токен (будь-який регістр: TENDERWIN…).
ПОПЕРЕДНІ_ТЕКИ = ('tenderwin',)
#: True — дозволити почати з порожньої бази, якщо попередньої не знайдено.
ПОЧАТИ_З_НУЛЯ = False
ПОТРІБНІ = ['tenderwin_lead_mail.py', 'tenderwin_lead_engine.py',
            'lead_machine_v1.py', 'tenderwin_service_cards.py']
БАЗА = 'tenderwin_lead_mail.db'
#: Без бази скрипт «забув» би, кому вже писав перший лист, і написав би ще раз.
СТАН = ['leads.db', '.gmail_token.json', БАЗА]      # база — останньою
ПОЗНАЧКА = '.перенесено_з_попередньої_теки.json'


def вміст(тека):
    try:
        return set(os.listdir(тека))
    except OSError:
        return set()


def лише_читання(шлях):
    """Зʼєднання, яке нічого не пише у файл, навіть журнал."""
    return sqlite3.connect(Path(шлях).as_uri() + '?mode=ro', uri=True)


def стан_бази(шлях):
    """'НЕМАЄ', 'ПОРОЖНЯ', 'ДАНІ' або 'ЗЛАМАНА' (файл є, але не читається)."""
    if not os.path.exists(шлях):
        return 'НЕМАЄ'
    зʼєднання = None
    try:
        зʼєднання = лише_читання(шлях)
        рядків = sum(зʼєднання.execute(f"SELECT COUNT(*) FROM {т}").fetchone()[0]
                     for т in ('companies', 'events', 'outreach', 'suppressions'))
        return 'ПОРОЖНЯ' if рядків == 0 else 'ДАНІ'
    except sqlite3.Error:
        return 'ЗЛАМАНА'
    finally:
        if зʼєднання is not None:
            зʼєднання.close()


def листи_в_базі(шлях):
    """[кількість листів, час останнього] або None, якщо не прочитали."""
    зʼєднання = None
    try:
        зʼєднання = лише_читання(шлях)
        кількість, останній = зʼєднання.execute(
            "SELECT COUNT(*), MAX(generated_at) FROM outreach").fetchone()
        return [кількість, останній or '']
    except sqlite3.Error:
        return None
    finally:
        if зʼєднання is not None:
            зʼєднання.close()


def скопіювати(звідки, куди):
    """Через тимчасову назву: обрив не лишить напівфайла під справжньою назвою.
    Журнал SQLite — частина бази; він копіюється першим, база — останньою."""
    for хвіст in ('-wal', '-journal', ''):
        джерело = звідки + хвіст
        if хвіст and not (os.path.exists(джерело) and os.path.getsize(джерело) > 0):
            continue
        тимчасова = куди + хвіст + '.part'
        try:
            shutil.copyfile(джерело, тимчасова)
            if os.path.getsize(тимчасова) != os.path.getsize(джерело):
                raise OSError(f"копія {os.path.basename(джерело)} неповна")
            os.replace(тимчасова, куди + хвіст)
        except BaseException:
            if os.path.exists(тимчасова):
                os.remove(тимчасова)
            raise


DYSK = os.path.join(КОРІНЬ, РОБОЧА_ТЕКА)
if not os.path.isdir(DYSK):
    os.makedirs(DYSK, exist_ok=True)
    print("Теки не було — створив:", DYSK)
print("Робоча тека:", DYSK)

# --- База, історія і токен: ОДИН раз із попередньої теки ----------------------
попередні = [os.path.join(КОРІНЬ, назва) for назва in sorted(os.listdir(КОРІНЬ))
             if назва.lower() in ПОПЕРЕДНІ_ТЕКИ
             and os.path.isdir(os.path.join(КОРІНЬ, назва))
             and os.path.abspath(os.path.join(КОРІНЬ, назва)) != os.path.abspath(DYSK)]
попередні.sort(key=lambda т: os.path.getsize(os.path.join(т, БАЗА))
               if os.path.exists(os.path.join(т, БАЗА)) else -1, reverse=True)
ДЖЕРЕЛО = next((т for т in попередні if os.path.exists(os.path.join(т, БАЗА))), None)

try:
    with open(os.path.join(DYSK, ПОЗНАЧКА), encoding='utf-8') as файл_позначки:
        позначка = json.load(файл_позначки)
except (OSError, ValueError):
    позначка = None

робоча_база = os.path.join(DYSK, БАЗА)
стан = стан_бази(робоча_база)
зупинка = ''

if позначка:
    # Перенесення вже було — більше нічого не копіюємо, навіть видалений токен.
    звідки_було = позначка.get('звідки') or ''
    if стан == 'НЕМАЄ':
        зупинка = (f"СТОП: із робочої теки зникла база {БАЗА}.\n"
                   f"   Поверніть її (Кошик Google Диска або тека, куди її перенесли).\n"
                   f"   Стару копію з {звідки_було} скрипт не підставляє: у ній немає\n"
                   f"   листів після перенесення ({позначка.get('коли', '?')}).")
    elif стан == 'ЗЛАМАНА':
        зупинка = (f"СТОП: база {робоча_база} не читається.\n"
                   f"   Відновіть її з історії версій файла на Google Диску.")
    elif стан == 'ПОРОЖНЯ' and (позначка.get('листів_у_джерелі') or [0])[0] > 0:
        зупинка = (f"СТОП: база в робочій теці порожня, хоча при перенесенні в ній\n"
                   f"   було листів: {позначка['листів_у_джерелі'][0]}. Поверніть справжню базу\n"
                   f"   (історія версій файла на Google Диску).")
    else:
        було = позначка.get('листів_у_джерелі')
        тепер = листи_в_базі(os.path.join(звідки_було, БАЗА)) if звідки_було else None
        if було and тепер and (тепер[0] > було[0] or тепер[1] > було[1]):
            print(f"УВАГА: у {звідки_було} після перенесення зʼявилися нові листи"
                  f" ({було[0]} → {тепер[0]}).")
            print("   Схоже, запускали старий ноутбук із тієї теки. Ці листи в робочу")
            print("   базу не потрапили — не відправляйте нових, доки бази не обʼєднано.")
elif стан in ('НЕМАЄ', 'ПОРОЖНЯ', 'ЗЛАМАНА') and ДЖЕРЕЛО:
    try:
        перенесено = []
        if стан != 'НЕМАЄ':
            # Порожню чи нечитабельну базу не видаляємо — відкладаємо поруч.
            хвіст = '.empty_' if стан == 'ПОРОЖНЯ' else '.broken_'
            відкладена = робоча_база + хвіст + time.strftime('%Y%m%d-%H%M%S')
            os.replace(робоча_база, відкладена)
            print(f"Наявну базу ({стан.lower()}) відкладено: {os.path.basename(відкладена)}")
        for назва in СТАН:
            звідки, куди = os.path.join(ДЖЕРЕЛО, назва), os.path.join(DYSK, назва)
            if os.path.exists(звідки) and (назва == БАЗА or not os.path.exists(куди)):
                скопіювати(звідки, куди)
                перенесено.append(назва)
        with open(os.path.join(DYSK, ПОЗНАЧКА), 'w', encoding='utf-8') as файл_позначки:
            json.dump({'звідки': ДЖЕРЕЛО, 'коли': time.strftime('%Y-%m-%d %H:%M:%S'),
                       'файли': перенесено,
                       'листів_у_джерелі': листи_в_базі(os.path.join(ДЖЕРЕЛО, БАЗА))},
                      файл_позначки, ensure_ascii=False, indent=1)
    except Exception:
        print("\nПеренесення не завершилося. Нічого не зіпсовано: у TENDERWIN усе як було,")
        print("а в робочій теці немає напівфайлів. Виконайте клітинку 2 ще раз.")
        raise
    print(f"Скопійовано з {ДЖЕРЕЛО}: " + ", ".join(перенесено))
    print("   Це одноразово: далі скрипт працює лише з робочою текою, а TENDERWIN")
    print("   лишається резервною копією. Старий ноутбук із TENDERWIN не запускайте.")
    if '.gmail_token.json' in перенесено:
        print("   ⓘ Токен пошти тепер і в цій теці. Надійніше тримати його в Colab")
        print("     Secrets (GMAIL_OAUTH_JSON) — тоді файл можна видалити, він не повернеться.")
elif стан in ('НЕМАЄ', 'ПОРОЖНЯ', 'ЗЛАМАНА'):
    if стан == 'ЗЛАМАНА':
        зупинка = f"СТОП: база {робоча_база} не читається, а попередньої бази не знайдено."
    elif not ПОЧАТИ_З_НУЛЯ:
        зупинка = ("СТОП: попередньої бази не знайдено (шукав теку TENDERWIN у корені\n"
                   "   «Мого диска»). Без неї скрипт не знатиме, кому вже писав.\n"
                   "   Якщо теку перейменовано — допишіть назву в ПОПЕРЕДНІ_ТЕКИ.\n"
                   "   Якщо справді починаєте з нуля — поставте ПОЧАТИ_З_НУЛЯ = True.")
elif ДЖЕРЕЛО and стан_бази(os.path.join(ДЖЕРЕЛО, БАЗА)) == 'ДАНІ':
    # У робочій теці вже є база з даними, а позначки немає.
    тут, там = листи_в_базі(робоча_база), листи_в_базі(os.path.join(ДЖЕРЕЛО, БАЗА))
    if тут and там and тут[0] >= там[0] and тут[1] >= там[1]:
        # Базу перенесли вручну (і, можливо, вже працювали): нічого не бракує.
        with open(os.path.join(DYSK, ПОЗНАЧКА), 'w', encoding='utf-8') as файл_позначки:
            json.dump({'звідки': ДЖЕРЕЛО, 'коли': time.strftime('%Y-%m-%d %H:%M:%S'),
                       'файли': [], 'листів_у_джерелі': там, 'вручну': True},
                      файл_позначки, ensure_ascii=False, indent=1)
    else:
        print(f"УВАГА: у робочій теці вже є своя база, а в {ДЖЕРЕЛО} — інша,")
        print("   і в ній є листи, яких тут немає. Скрипт жодну не перезаписує.")
        print("   Не відправляйте листи (клітинка 7), доки бази не обʼєднано.")

# --- Модулі лише з робочої теки ---------------------------------------------
# Якщо в цьому сеансі вже працювали з іншою текою, Python памʼятає її модулі
# і reload перечитав би старі файли. Забуваємо їх і ставимо робочу теку першою.
for назва_модуля in [н[:-3] for н in ПОТРІБНІ]:
    модуль = sys.modules.get(назва_модуля)
    файл = getattr(модуль, '__file__', '') or ''
    if модуль is not None and os.path.dirname(os.path.abspath(файл)) != os.path.abspath(DYSK):
        del sys.modules[назва_модуля]
sys.path[:] = [шлях for шлях in sys.path
               if not os.path.exists(os.path.join(шлях or '.', 'tenderwin_lead_mail.py'))]
sys.path.insert(0, DYSK)


def чого_бракує():
    файли = вміст(DYSK)
    return [назва for назва in ПОТРІБНІ if назва not in файли]


def справжня_назва(назва):
    # Браузер часто додає "(1)": tenderwin_lead_mail (1).py
    чиста = re.sub(r"\s*\(\d+\)(?=\.py$)", "", назва).strip()
    return чиста if чиста in ПОТРІБНІ else назва


if зупинка:
    print("\n" + зупинка)
else:
    бракує = чого_бракує()
    if not бракує:
        print("Усі файли на місці. Далі — клітинка 3.")
    else:
        print("Бракує:", ", ".join(бракує))
        print("Оберіть ці файли на компʼютері у вікні нижче.")
        print("(Або покладіть їх у цю теку на Диску і виконайте клітинку ще раз.)")
        try:
            from google.colab import files
            завантажені = files.upload()
            for тимчасова in list(завантажені):
                назва = справжня_назва(тимчасова)
                if назва in ПОТРІБНІ:
                    куди = os.path.join(DYSK, назва)
                    if os.path.exists(куди):
                        os.remove(куди)
                    shutil.move(тимчасова, куди)
                    print("  покладено на Диск:", назва)
                else:
                    print("  пропущено (не потрібен тут):", тимчасова)
        except Exception as помилка:
            print("Вікно вибору не відкрилося:", type(помилка).__name__)
        бракує = чого_бракує()
        print()
        print("Усі файли на місці. Далі — клітинка 3." if not бракує
              else "Досі бракує: " + ", ".join(бракує) + ". Виконайте клітинку ще раз.")
    ГОТОВО = not бракує

## КЛІТИНКА 3 · Доступ до пошти

Секрет `GMAIL_OAUTH_JSON` має бути в Colab Secrets (ключик ліворуч), доступ
для цього ноутбука — увімкнений. У код секрети не вставляються ніколи.

In [ ]:
import sys, os

if not globals().get("ГОТОВО") or "РОБОЧА_ТЕКА" not in globals():
    print("Спершу виконайте клітинку 2 — вона має закінчитися словами «Усі файли на місці».")
    ФАЙЛИ_Є = False
else:
    if DYSK not in sys.path:
        sys.path.insert(0, DYSK)
    try:
        import tenderwin_lead_engine as E
        ФАЙЛИ_Є = True
    except ModuleNotFoundError as помилка:
        ФАЙЛИ_Є = False
        print("Немає файла:", (помилка.name or "?") + ".py")
        print("Поверніться до клітинки 2 — вона покладе файли на Диск.")

if ФАЙЛИ_Є:
    # Без цього рядка токен, що лежить файлом на Диску, просто не видно:
    # движок шукає .gmail_token.json у теці, яку йому назвали.
    E.TOKEN_DIR = DYSK
    у_теці = os.path.exists(os.path.join(DYSK, E.TOKEN_FILE_NAME))

    if E._get_secret(E.GMAIL_OAUTH_ENV):
        print("Доступ до пошти є" + (f" (файл {E.TOKEN_FILE_NAME} у теці на Диску)."
                                     if у_теці else " (секрет Colab)."))
        бракує_дозволів = E.missing_scopes()
        if бракує_дозволів:
            print("  Але токену бракує дозволів:", ", ".join(бракує_дозволів))
            print("  Зробіть E.reset_gmail_token(DYSK), потім E.mint_oauth_token().")
        else:
            print("Далі — клітинка 4.")
        if у_теці:
            print("  ⓘ Токен у теці на Диску може прочитати кожен, хто має")
            print("    доступ до цієї теки. Надійніше тримати його в Colab Secrets.")
    else:
        print(f"Доступу до пошти немає: ні секрету {E.GMAIL_OAUTH_ENV},")
        print(f"ні файла {E.TOKEN_FILE_NAME} у теці {DYSK}.")
        print("Без нього працює сухий прогін (клітинка 6), відправки не буде.")
        print("Як отримати токен: E.mint_oauth_token()")

## КЛІТИНКА 4 · Завантаження скрипта

Перевіряє, що на Диску саме версії 1.5.0 і 1.1.1 (службові картки). Якщо там старий файл —
зупиняється і пише, який саме файл замінити.

In [ ]:
import importlib, os

# Версії, з якими працює цей ноутбук. Старіший файл на Диску — клітинка
# зупиниться й назве його, а не впаде далі з незрозумілою помилкою.
ПОТРІБНІ_ВЕРСІЇ = {"tenderwin_lead_mail": "1.5.0", "tenderwin_service_cards": "1.1.1"}


def номер(версія):
    try:
        return tuple(int(частина) for частина in str(версія).split("."))
    except ValueError:
        return (0,)


M = None
# Перезавантаження модуля повертає налаштування за замовчуванням (режим
# TEST): після цієї клітинки обовʼязково клітинка 5.
НАЛАШТОВАНО = False
if not globals().get("ГОТОВО") or "РОБОЧА_ТЕКА" not in globals():
    print("Спершу виконайте клітинку 2 — вона має закінчитися словами «Усі файли на місці».")
else:
    модулі = {}
    for назва in ПОТРІБНІ_ВЕРСІЇ:
        try:
            модуль = importlib.import_module(назва)
            модулі[назва] = importlib.reload(модуль)
        except ModuleNotFoundError as помилка:
            print("Немає файла:", (помилка.name or назва) + ".py")
            print("Поверніться до клітинки 2.")
            модулі = None
            break
    if модулі:
        застарілі = []
        for назва, треба in ПОТРІБНІ_ВЕРСІЇ.items():
            модуль = модулі[назва]
            є = getattr(модуль, "VERSION", "?")
            if номер(є) < номер(треба):
                шлях = getattr(модуль, "__file__", "") or ""
                розмір = os.path.getsize(шлях) if шлях and os.path.exists(шлях) else 0
                застарілі.append((назва, є, треба, шлях, розмір))
        if застарілі:
            print("СТОП: на Диску стара версія скрипта.")
            for назва, є, треба, шлях, розмір in застарілі:
                print(f"   {назва}.py — версія {є}, а потрібна {треба}")
                print(f"      файл: {шлях} ({розмір:,} байтів)".replace(",", " "))
            print()
            print("Що зробити:")
            print(f"  1. Покладіть новий файл із пакета 1.5.0 у теку {DYSK}")
            print("     (у вікні Google Диска оберіть «Замінити наявний файл»,")
            print("     а не «Зберегти обидва»).")
            print("  2. Меню «Середовище виконання» → «Перезапустити сеанс».")
            print("  3. Клітинки 2, 3, 4 ще раз.")
        else:
            M = модулі["tenderwin_lead_mail"]
            print(f"tenderwin_lead_mail {M.VERSION} · "
                  f"tenderwin_service_cards {модулі['tenderwin_service_cards'].VERSION}"
                  f" · з теки {os.path.dirname(M.__file__)}")

if M is not None:
    print("Далі — клітинка 5: налаштування і підготовка бази.")

## КЛІТИНКА 5 · Налаштування і підготовка бази

Значення нижче — чинне рішення власника. **`M.MODE = "LIVE"`** — бойовий
режим: листи складаються на адреси компаній. Щоб перевірити щось без
клієнтів, поставте `"TEST"` — тоді листи підуть лише на ваші тестові скриньки.

Після налаштувань клітинка готує базу (`M.setup`). Клітинки 6–8 працюють лише
після неї: якщо перезапустити клітинку 4, виконайте і клітинку 5.

In [ ]:
if globals().get("M") is None:
    print("Спершу виконайте клітинки 2–4.")
else:
    M.MODE           = "LIVE"           # БОЙОВИЙ: листи клієнтам. "TEST" — на ваші скриньки
    M.LETTER_VERSION = "V10_OWNER"      # ЧИННИЙ лист (рішення 08.10.2026)
    M.LETTER_FORMAT  = "HTML_AND_TEXT"  # "TEXT_ONLY" — лист лише текстом
    M.WINDOW_MODE    = "TODAY_ONLY"     # лише поточна доба
    M.SKIP_WITHOUT_COMPLAINT_ROUTE = True   # без строку оскарження — не пишемо
    M.MIN_LOT_VALUE  = 0                # 0 = без порогу вартості
    M.MAX_SENDS_PER_RUN = 25            # скільки листів за один запуск клітинки 7
    M.SERVICE_CARDS  = True             # службові картки з протоколами
    M.SERVICE_CARDS_FRESH = True        # свіжий стан рішення з Prozorro
    M.CASES_DIR = ""                    # "" = «Мій диск/0 Cases» (G:\Мій диск\0 Cases)

    print("режим:", M.MODE, "· лист:", M.LETTER_VERSION, "·", M.LETTER_FORMAT)
    if M.MODE == "LIVE":
        print("БОЙОВИЙ РЕЖИМ: листи складаються на адреси компаній із Prozorro.")
        print("   Надсилає клітинка 7 — одразу, без запитань.")
    else:
        print("тестовий режим · адрес для тестової доставки:", len(M.TEST_RECIPIENTS))
    print("службові картки:", "так" if M.SERVICE_CARDS else "ні")
    if M.SERVICE_CARDS:
        print("справи (картка + протокол):", M.cases_root())
    print()
    M.setup(DYSK)      # база, історія попереднього движка — уже в обраному режимі
    НАЛАШТОВАНО = True

## КЛІТИНКА 6 · Сухий прогін

Шукає відхилення за сьогодні, складає листи у файли і робочі картки.
**Нічого нікуди не надсилає** — і в бойовому режимі теж.

У теці `vyhid_lystiv/eml` біля кожного листа лежить файл `.html` — відкрийте
його у браузері, щоб побачити лист таким, яким його побачить клієнт. У файлі
`.txt` видно, на яку адресу компанії піде лист.

In [ ]:
if globals().get("M") is None or not globals().get("НАЛАШТОВАНО"):
    print("Спершу виконайте клітинки 2–5: без клітинки 5 режим був би TEST, а не LIVE.")
else:
    результат = M.go(dysk=DYSK)

## КЛІТИНКА 7 · Відправка і стан бази

**Бойовий режим:** листи підуть **клієнтам** — на адреси компаній із Prozorro,
**одразу, без запитань**. Якщо натиснути «Виконати всі», клітинка 7 спрацює
відразу після клітинки 6 — без паузи на перегляд листів.

Між листами — пауза 2–3 хвилини; за один запуск — до `M.MAX_SENDS_PER_RUN`
листів (клітинка 5). Якщо в черзі лишились листи, клітинка скаже про це —
запустіть її ще раз. Під час відправки видно кожен рядок «→ адреса».

Надішліть чергу **того самого дня**: завтра клітинка 6 складе листи лише за
завтрашні відхилення. Повторний запуск клітинки 6 сьогодні нічого не губить —
ненадіслані листи складаються заново.

Якщо Gmail відмовить (ліміт, токен), лист лишається в черзі (до 3 спроб), а
після двох відмов поспіль відправка зупиняється — запустіть клітинку 7 пізніше.

**Тестовий режим:** листи підуть лише на адреси зі списку `TEST_RECIPIENTS`.

В обох режимах база окремо від коду перевіряє адресу: бойовий лист — лише на
адресу самої компанії, тестовий — лише на тестову скриньку.

In [ ]:
if globals().get("M") is None or not globals().get("НАЛАШТОВАНО"):
    print("Спершу виконайте клітинки 2–5: без клітинки 5 режим був би TEST, а не LIVE.")
else:
    M.send(dysk=DYSK)     # LIVE: листи клієнтам, одразу
    M.state(dysk=DYSK)    # що зараз у базі

## КЛІТИНКА 8 · Службові картки ще раз (за потреби)

Після `M.go()` картки вже зроблені: кожна справа — окрема тека в
`Мій диск / 0 Cases` з назвою «ID закупівлі + ЄДРПОУ (ІПН) учасника», у ній
службова картка і протокол (рішення) замовника. Ця клітинка потрібна лише
тоді, коли крок карток обірвався або коли треба зібрати картки за інший день.

Уже завантажені файли вдруге не качаються. Картку, у якій ви щось дописали,
скрипт **не перезаписує** — нова версія ляже поруч окремим файлом.

In [ ]:
if globals().get("M") is None or not globals().get("НАЛАШТОВАНО"):
    print("Спершу виконайте клітинки 2–5: без клітинки 5 режим був би TEST, а не LIVE.")
else:
    M.kartky(dysk=DYSK)                      # події останнього прогону
    # M.kartky(day="2026-10-02", dysk=DYSK)  # усі події конкретного дня

## КЛІТИНКА 9 · Перевірка самого скрипта (за бажанням)

Запускає тести з теки `testy` робочої теки на вигаданих даних: без мережі,
без листів, робоча база не чіпається. Займає близько хвилини.

In [ ]:
import os, subprocess, sys

if not globals().get("ГОТОВО") or "РОБОЧА_ТЕКА" not in globals():
    print("Спершу виконайте клітинку 2.")
else:
    тека_тестів = os.path.join(DYSK, "testy")
    for тест in ("test_lead_mail.py", "test_0_cases.py", "test_live.py"):
        шлях = os.path.join(тека_тестів, тест)
        if not os.path.exists(шлях):
            print(f"  {тест}: немає у {тека_тестів}")
            continue
        # Модулі — з робочої теки, тести — з testy.
        результат = subprocess.run(
            [sys.executable, шлях], capture_output=True, text=True, cwd=тека_тестів,
            env={**os.environ, "PYTHONPATH": DYSK})
        рядки = (результат.stderr or результат.stdout).strip().splitlines()
        print(f"  {тест}: " + ("OK" if результат.returncode == 0 else "Є ПОМИЛКИ")
              + " · " + " · ".join(рядки[-3:]))

---
### Якщо щось пішло не так

| Повідомлення | Що робити |
|---|---|
| `Немає файла: …py` | клітинка 2 — покладіть файли на Диск |
| `СТОП: на Диску стара версія скрипта` | замініть названий файл у робочій теці новим із пакета 1.5.0, перезапустіть сеанс, клітинки 2–4 |
| `module 'tenderwin_lead_mail' has no attribute …` | те саме: Colab бачить старий файл — див. рядок вище |
| `СТОП: із робочої теки зникла база …` | поверніть `tenderwin_lead_mail.db` у робочу теку (Кошик Диска); стару копію з TENDERWIN скрипт сам не підставляє |
| `СТОП: попередньої бази не знайдено` | теку TENDERWIN перейменовано чи перенесено — допишіть назву в `ПОПЕРЕДНІ_ТЕКИ` (клітинка 2) |
| `УВАГА: … нові листи` або `… своя база, а в … інша` | дві бази розійшлися: листи не відправляти, доки їх не обʼєднано |
| `Перенесення не завершилося` | обрив копіювання; нічого не зіпсовано — клітинка 2 ще раз |
| `TRANSPORT_UNAVAILABLE` | немає секрету `GMAIL_OAUTH_JSON` — клітинка 3 |
| `TEST_ALLOWLIST_EMPTY` | лише в режимі TEST: порожній `TEST_RECIPIENTS` — заповніть |
| `LIVE_DELIVERY_NOT_CONTACT` | запобіжник бази: бойовий лист не на адресу компанії — цей лист не складено, решта працює |
| `MODE_UNKNOWN` | у клітинці 5 має бути `M.MODE = "LIVE"` або `"TEST"` |
| `У черзі листи від …, а сьогодні …` | черга вчорашня; клітинка 6 складе сьогоднішні листи заново |
| `PROZORRO_UNAVAILABLE` | Prozorro не відповідає — спробуйте за кілька хвилин |
| `Обхід стрічки: INCOMPLETE` | прочитано не всю стрічку; у звіті написано чому |
| `листи без певного стану` | зробіть `M.send(dysk=DYSK)` — він звірить їх із Gmail |
| `SERVICE_CARDS_MODULE_MISSING` | немає `tenderwin_service_cards.py` — клітинка 2; листи цим не зачеплені |
| `SERVICE_CARDS_FAILED` | крок карток обірвався; листи збережено. Повторіть `M.kartky(dysk=DYSK)` |

**Де службові картки і протоколи:** `Мій диск → 0 Cases → <UA-ID> <ЄДРПОУ/ІПН>`
(на компʼютері `G:\Мій диск\0 Cases`). Якщо того самого учасника в тій самій
закупівлі відхилили двічі (два лоти, повторне рішення), друга справа — у теці
з « (2)» в кінці.

**Перелік для відбору:** `1 Lead&Mail generator → Службові картки → <дата> →
_ПЕРЕЛІК_<прогін>.xlsx` — один рядок на подію, колонка «Тека справи».

Перевірити сам скрипт: клітинка 9.